Базовая модель для кандидатогенерации. Лексический поиск BM25 (с лемматизацией) + бусты за совпадение локации и популярность.



In [26]:
import re
import time
import gc
from functools import lru_cache

import numpy as np
import pandas as pd
from scipy import sparse  # noqa: F401
from sklearn.feature_extraction.text import CountVectorizer

Пути к файлам и параметры поиска.

In [ ]:
TRAIN_PATH = "../data/train.parquet"
BENCH_QUERIES_PATH = "../data/benchmark_queries.parquet"
BENCH_ITEMS_PATH = "../data/benchmark_items.parquet"
OUTPUT_PATH = "../baseline_output/answer.csv"

TOP_K = 50

NORMALIZE_SCORES = True
LOCATION_BOOST = 1.0       
POPULARITY_BOOST = 0.0      
TEXT_QUOTA = 0              
QUERY_USE_FILTERS = True    

QUICK_TEST = False
QUICK_TRAIN_ROWS = 30_000
QUICK_BENCH_ITEMS = 10_000
QUICK_VAL_QUERIES = 200

FULL_BENCH_PADDING = None 
FULL_VAL_QUERIES = 1000

TRAIN_COLS = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
    "item_id", "item_title_raw", "item_description_raw",
    "item_infm_params_text", "item_category_id", "item_location_id",
    "item_rating_reviews_count",
]
ITEM_COLS = [
    "item_id", "item_title_raw", "item_description_raw",
    "item_infm_params_text", "item_category_id", "item_location_id",
    "item_rating_reviews_count",
]

## Загрузка train

Читаем только нужные колонки — экономим память. В быстром режиме берём 
первые `QUICK_TRAIN_ROWS` строк.

In [ ]:
def load_train(n_rows=None) -> pd.DataFrame:
    if n_rows is None:
        return pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLS)
    import pyarrow.parquet as pq
    first_batch = next(pq.ParquetFile(TRAIN_PATH).iter_batches(batch_size=n_rows, columns=TRAIN_COLS))
    return first_batch.to_pandas()

## Предобработка текста

Токенизация: нижний регистр, только буквы/цифры (русские и латинские).

Лемматизация через `pymorphy3`: «бани», «баню» → «баня». Без неё BM25 считает 
эти формы разными словами и не находит совпадения.

In [ ]:
TOKEN_RE = re.compile(r"[a-zа-яё0-9]+", re.IGNORECASE)

LEMMATIZE = True

try:
    import pymorphy3
    _MORPH = pymorphy3.MorphAnalyzer()
except ImportError:
    _MORPH = None

_CYRILLIC_RE = re.compile(r"[а-яё]+")


@lru_cache(maxsize=None)
def lemma(word: str) -> str:
    """Начальная форма слова. Результат кэшируется: каждое уникальное слово
    разбирается один раз, а не при каждой встрече в 500k текстах."""
    if not _CYRILLIC_RE.fullmatch(word):      
        return word
    if _MORPH is None:
        raise ImportError("Для лемматизации нужен pymorphy3: pip install pymorphy3")
    return _MORPH.parse(word)[0].normal_form


def tokenize(text: str) -> list[str]:
    """Превращает строку в список нормализованных токенов."""
    if not isinstance(text, str) or not text:
        return []
    words = TOKEN_RE.findall(text.lower())
    if LEMMATIZE:
        words = [lemma(w) for w in words]
    return words


def build_item_text(row) -> str:
    """
    «Документ» для объявления. Заголовок дублируем (в нём сконцентрирован 
    смысл). Описание обрезаем до 300 символов, чтобы не размывать индекс.
    """
    title = str(row.get("item_title_raw", "") or "")
    params = str(row.get("item_infm_params_text", "") or "")
    desc = str(row.get("item_description_raw", "") or "")[:300]
    return f"{title} {title} {params} {desc}"


def build_query_text(row, use_filters: bool = True) -> str:
    """Текст запроса: сам запрос (+ текст фильтров, если use_filters=True)."""
    q = str(row.get("search_query", "") or "")
    if not use_filters:
        return q
    params = str(row.get("search_infm_params_text", "") or "")
    return f"{q} {params}"

## Проверка входных данных

Несовпадение типов колонок (pandas молча вернёт False при сравнении `int64 == object`), 
пропуски, дубликаты, неправильный формат `item_id`.

In [30]:
def sanity_check_inputs(queries_df: pd.DataFrame, items_df: pd.DataFrame) -> None:
    """Проверки колонок, типов, пропусков, дубликатов, формата item_id."""
    print("=== Проверка колонок ===")
    print("Колонки queries:", list(queries_df.columns))
    print("Колонки items:  ", list(items_df.columns))

    for q_col, i_col in [
        ("search_category", "item_category_id"),
        ("search_location_id", "item_location_id"),
    ]:
        if q_col in queries_df.columns and i_col in items_df.columns:
            q_dtype = queries_df[q_col].dropna().dtype
            i_dtype = items_df[i_col].dropna().dtype
            status = "OK" if q_dtype == i_dtype else "!!! НЕ СОВПАДАЮТ"
            print(f"{q_col} ({q_dtype})  vs  {i_col} ({i_dtype})  -> {status}")
            if q_dtype != i_dtype:
                print(f"    Пример {q_col}: {queries_df[q_col].dropna().iloc[0]!r}")
                print(f"    Пример {i_col}: {items_df[i_col].dropna().iloc[0]!r}")

    print("\n=== Примеры сырого текста ===")
    if "search_query" in queries_df.columns:
        print("search_query:", queries_df["search_query"].dropna().iloc[:5].tolist())
    if "item_title_raw" in items_df.columns:
        print("item_title_raw:", items_df["item_title_raw"].dropna().iloc[:5].tolist())

    print("\n=== Доля пропусков (NaN) в текстовых полях ===")
    for col in ["item_title_raw", "item_description_raw", "item_infm_params_text"]:
        if col in items_df.columns:
            share = items_df[col].isna().mean()
            print(f"{col}: {share:.1%} пропусков")

    if "item_id" in items_df.columns:
        n_dupes = items_df["item_id"].duplicated().sum()
        print(f"\nДубликаты item_id в items: {n_dupes}")

    if "item_id" in items_df.columns:
        bad_format = ~items_df["item_id"].astype(str).str.fullmatch(r"[0-9a-f]{16}")
        print(f"item_id неправильного формата: {bad_format.sum()} из {len(items_df)}")

## BM25

Реализация BM25 на разреженной матрице. Формула та же, что в `rank_bm25`, 
но вклад каждого слова считается **один раз** при построении индекса. Скор 
запроса = сумма нескольких столбцов матрицы (~1 мс) вместо питоновского цикла 
по 500k документам (~1 сек).

In [ ]:
class SparseBM25:
    """BM25 Okapi на разреженной матрице. k1=1.5, b=0.75 — классические значения."""

    def __init__(self, tokenized_corpus, k1=1.5, b=0.75, epsilon=0.25):
        self.vec = CountVectorizer(analyzer=lambda toks: toks)
        tf = self.vec.fit_transform(tokenized_corpus).tocsr().astype(np.float32)
        n_docs = tf.shape[0]
        df = np.diff(tf.tocsc().indptr)                      # в скольких документах слово
        idf = np.log(n_docs - df + 0.5) - np.log(df + 0.5)   # редкие слова весят больше
        idf = np.where(idf < 0, epsilon * idf.mean(), idf).astype(np.float32)
        dl = np.asarray(tf.sum(axis=1)).ravel()              # длина документа
        norm = k1 * (1 - b + b * dl / dl.mean())             # поправка на длину
        rows = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
        tfv = tf.data
        tf.data = idf[tf.indices] * tfv * (k1 + 1) / (tfv + norm[rows])
        self.W = tf.tocsc()                                  # быстрые срезы по столбцам
        self.vocab = self.vec.vocabulary_

    def get_scores(self, query_tokens):
        counts = {}
        for t in query_tokens:
            j = self.vocab.get(t)
            if j is not None:
                counts[j] = counts.get(j, 0) + 1
        if not counts:
            return np.zeros(self.W.shape[0], dtype=np.float32)
        cols = np.fromiter(counts.keys(), dtype=np.int64)
        w = np.fromiter(counts.values(), dtype=np.float32)
        return np.asarray(self.W[:, cols] @ w).ravel()

## Основной класс поиска

Обёртка над BM25. Строит индекс один раз, потом для каждого запроса возвращает 
топ-K id.

Дополнительные сигналы:
- **локация** (`search_location_id == item_location_id`): у ~80% нужных объявлений 
  совпадает — сильный сигнал;
- **популярность** (`log(1 + число отзывов)`, шкала 0..1): из одинаково подходящих 
  по тексту пользователи чаще выбирают популярные.

In [ ]:
class BM25CandidateGenerator:
    """BM25 + бусты локации и популярности."""

    def __init__(
        self,
        items_df: pd.DataFrame,
        location_boost: float = LOCATION_BOOST,
        popularity_boost: float = POPULARITY_BOOST,
        text_quota: int = TEXT_QUOTA,
        use_filters: bool = QUERY_USE_FILTERS,
        normalize: bool = NORMALIZE_SCORES,
    ):
        self.items_df = items_df.reset_index(drop=True)
        self.location_boost = location_boost
        self.popularity_boost = popularity_boost
        self.text_quota = text_quota
        self.use_filters = use_filters
        self.normalize = normalize
        self.item_ids = self.items_df["item_id"].values

        t0 = time.time()
        texts = self.items_df.apply(build_item_text, axis=1)
        print(f"  [время] сборка текста объявлений: {time.time() - t0:.1f} сек "
              f"({len(self.items_df)} объявлений)")

        t0 = time.time()
        tokenized_corpus = [tokenize(t) for t in texts]
        print(f"  [время] токенизация: {time.time() - t0:.1f} сек")

        t0 = time.time()
        self.bm25 = SparseBM25(tokenized_corpus)
        print(f"  [время] построение BM25-индекса: {time.time() - t0:.1f} сек")

        self.item_locations = self.items_df["item_location_id"].values

        if "item_rating_reviews_count" in self.items_df.columns:
            rc = self.items_df["item_rating_reviews_count"].fillna(0).clip(lower=0).to_numpy(dtype=np.float64)
            lg = np.log1p(rc)
            self.item_pop = (lg / max(lg.max(), 1e-9)).astype(np.float32)
        else:
            self.item_pop = np.zeros(len(self.items_df), dtype=np.float32)

    def score_parts(self, query_row):
        """(base, loc_mask, pop): чистый BM25, маска локации, популярность."""
        tokens = tokenize(build_query_text(query_row, self.use_filters))
        if not tokens:
            return None
        base = self.bm25.get_scores(tokens)
        if self.normalize:
            top = base.max()
            if top > 0:
                base = base / top
        sl = query_row.get("search_location_id")
        if pd.notna(sl):
            loc = (self.item_locations == sl).astype(np.float32)
        else:
            loc = np.zeros(len(self.items_df), np.float32)
        return base, loc, self.item_pop

    @staticmethod
    def top_index(scores, top_k: int = TOP_K):
        """Индексы top_k лучших объявлений по убыванию скора."""
        top_k = min(top_k, len(scores))
        idx = np.argpartition(scores, -top_k)[-top_k:]
        return idx[np.argsort(scores[idx])[::-1]]

    def pick(self, base, loc, pop, top_k=TOP_K, loc_w=None, pop_w=None, quota=None):
        loc_w = self.location_boost if loc_w is None else loc_w
        pop_w = self.popularity_boost if pop_w is None else pop_w
        quota = self.text_quota if quota is None else quota
        main = self.top_index(base + loc * loc_w + pop * pop_w, top_k)
        if quota <= 0:
            return main
        keep = main[: top_k - quota]
        seen = set(keep.tolist())
        text_rank = self.top_index(base + pop * pop_w, top_k)
        extra = [k for k in text_rank.tolist() if k not in seen][:quota]
        return np.concatenate([keep, np.array(extra, dtype=keep.dtype)])

    def search(self, query_row, top_k: int = TOP_K) -> list[str]:
        parts = self.score_parts(query_row)
        if parts is None:
            return list(self.item_ids[:top_k])
        base, loc, pop = parts
        return list(self.item_ids[self.pick(base, loc, pop, top_k)])

## Метрика и подготовка валидации

`Recall@K` — доля релевантных объявлений, попавших в топ-K.

Валидация: 15% уникальных запросов train (ключ = query+location+delivery+params+category) 
откладываются. В корпус добавляются объявления из `benchmark_items` — 
иначе плотность корпуса ниже реальной, и Recall завышен.

In [33]:
def recall_at_k(predictions: dict, relevant: dict) -> float:
    """Средний Recall@K по всем запросам."""
    scores = []
    for qid, rel_items in relevant.items():
        if not rel_items:
            continue
        pred_set = set(predictions.get(qid, []))
        hit = len(pred_set & rel_items) / len(rel_items)
        scores.append(hit)
    return float(np.mean(scores)) if scores else 0.0


def recall_per_query(predictions: dict, relevant: dict) -> dict:
    """Recall@K по каждому запросу — для парного сравнения конфигураций."""
    out = {}
    for qid, rel_items in relevant.items():
        if rel_items:
            out[qid] = len(set(predictions.get(qid, [])) & rel_items) / len(rel_items)
    return out


def prepare_validation(bench_items_df: pd.DataFrame | None = None):
    """
    Возвращает (items_pool, query_features, relevant).
    Делит train на запросы и релевантные объявления, откладывает 15% запросов,
    добавляет benchmark_items как «статистов» для честной плотности корпуса.
    """
    t0 = time.time()
    train = load_train(QUICK_TRAIN_ROWS if QUICK_TEST else None)
    print(f"  [время] чтение train.parquet: {time.time() - t0:.1f} сек ({len(train)} строк)"
          f"{'  [БЫСТРЫЙ РЕЖИМ]' if QUICK_TEST else ''}")

    t0 = time.time()
    train["_query_key"] = (
        train["search_query"].astype(str) + "|" +
        train["search_location_id"].astype(str) + "|" +
        train["search_is_delivery_search"].astype(str) + "|" +
        train["search_infm_params_text"].astype(str) + "|" +
        train["search_category"].astype(str)
    )
    print(f"  [время] построение _query_key: {time.time() - t0:.1f} сек")

    unique_queries = np.array(train["_query_key"].unique(), dtype=object)
    rng = np.random.default_rng(42)
    rng.shuffle(unique_queries)

    n_val_cap = QUICK_VAL_QUERIES if QUICK_TEST else FULL_VAL_QUERIES
    n_val = min(max(1, int(0.15 * len(unique_queries))), n_val_cap)
    val_keys = set(unique_queries[:n_val])

    val_rows = train[train["_query_key"].isin(val_keys)]
    relevant = val_rows.groupby("_query_key")["item_id"].apply(set).to_dict()
    query_features = val_rows.drop_duplicates("_query_key").set_index("_query_key")

    t0 = time.time()
    train_items_pool = train.drop_duplicates("item_id")[ITEM_COLS].copy()
    train_items_pool["item_description_raw"] = (
        train_items_pool["item_description_raw"].astype(str).str.slice(0, 300)
    )

    del train, val_rows
    gc.collect()

    if bench_items_df is None:
        bench_items_df = pd.read_parquet(BENCH_ITEMS_PATH, columns=ITEM_COLS)
    n_padding = QUICK_BENCH_ITEMS if QUICK_TEST else FULL_BENCH_PADDING
    if n_padding is not None and n_padding < len(bench_items_df):
        bench_items_df = bench_items_df.sample(n_padding, random_state=42)
    bench_items_for_padding = bench_items_df[ITEM_COLS].copy()
    bench_items_for_padding["item_description_raw"] = (
        bench_items_for_padding["item_description_raw"].astype(str).str.slice(0, 300)
    )

    already_have = set(train_items_pool["item_id"])
    bench_items_for_padding = bench_items_for_padding[
        ~bench_items_for_padding["item_id"].isin(already_have)
    ]
    items_pool = pd.concat([train_items_pool, bench_items_for_padding], ignore_index=True)
    del bench_items_for_padding, already_have
    gc.collect()

    print(f"  [время] сборка items_pool: {time.time() - t0:.1f} сек")
    print(f"Размер корпуса для валидации: {len(items_pool)} "
          f"(train: {len(train_items_pool)}, добавлено из benchmark: "
          f"{len(items_pool) - len(train_items_pool)})")

    return items_pool, query_features, relevant

## Сравнение «без лемматизации» / «с лемматизацией»

Честное сравнение: одинаковые запросы, одинаковый корпус, одинаковые бусты. 
Меняется только флаг `LEMMATIZE`.

Используется **bootstrap**: пересэмплируем запросы с возвращением 2000 раз, 
смотрим 95% доверительный интервал разницы. Если интервал не включает 0 — 
улучшение значимо.

In [34]:
def run_local_validation(bench_items_df: pd.DataFrame | None = None):
    """Сравнение «без лемматизации» / «с лемматизацией» на одной валидации."""
    items_pool, query_features, relevant = prepare_validation(bench_items_df)

    global LEMMATIZE
    original_flag = LEMMATIZE
    per_query = {}
    for use_lemmas in (False, True):
        LEMMATIZE = use_lemmas
        name = "с лемматизацией" if use_lemmas else "без лемматизации"
        print(f"\n=== Прогон: {name} ===")
        generator = BM25CandidateGenerator(items_pool)

        t0 = time.time()
        predictions = {}
        for qkey, qrow in query_features.iterrows():
            predictions[qkey] = generator.search(qrow, top_k=TOP_K)
        print(f"  [время] поиск по {len(query_features)} запросам: {time.time() - t0:.1f} сек")

        per_query[use_lemmas] = recall_per_query(predictions, relevant)
        print(f"  Recall@{TOP_K} ({name}) = {np.mean(list(per_query[use_lemmas].values())):.4f}")
        del generator
        gc.collect()
    LEMMATIZE = original_flag

    keys = list(per_query[False].keys())
    a = np.array([per_query[False][k] for k in keys])
    b = np.array([per_query[True][k] for k in keys])
    diff = b - a
    rng_b = np.random.default_rng(0)
    boots = [diff[rng_b.integers(0, len(diff), len(diff))].mean() for _ in range(2000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])

    print("\n================ ИТОГ СРАВНЕНИЯ ================")
    print(f"Запросов в валидации:          {len(keys)}")
    print(f"Recall@{TOP_K} без лемматизации: {a.mean():.4f}")
    print(f"Recall@{TOP_K} с лемматизацией:  {b.mean():.4f}")
    print(f"Разница:                       {diff.mean():+.4f}  (95% интервал: {lo:+.4f} .. {hi:+.4f})")
    print(f"Запросов лучше / хуже / так же: {(diff > 0).sum()} / {(diff < 0).sum()} / {(diff == 0).sum()}")
    if lo > 0:
        print("Вывод: лемматизация помогает (интервал целиком выше нуля).")
    elif hi < 0:
        print("Вывод: лемматизация вредит (интервал целиком ниже нуля).")
    else:
        print("Вывод: разница в пределах шума, уверенно сказать нельзя.")
    return {"no_lemma": float(a.mean()), "lemma": float(b.mean())}

## Эксперименты и диагностика

- `evaluate_grid` — перебор весов локации и популярности.
- `evaluate_quota` — перебор размера квоты «чистый текст без локации».
- `run_diagnostics` — где стоит нужное объявление среди всех (топ-50 / 51–200 / 201–1000 / >1000), доля пар без общих слов, доля совпадения локации.
- `dump_examples` — примеры промахов и находок в CSV.
- `run_experiments` — обёртка: прогоняет всё и печатает лучшие параметры.

In [35]:
def run_diagnostics(generator, queries_df, relevant, loc_boost, pop_boost):
    """Отвечает на вопрос 'почему не находим': ранги нужных объявлений."""
    id2idx = {i: k for k, i in enumerate(generator.item_ids)}
    ranks, zero_overlap, loc_ok, n_pairs = [], 0, 0, 0
    for qkey, qrow in queries_df.iterrows():
        parts = generator.score_parts(qrow)
        for item_id in relevant[qkey]:
            n_pairs += 1
            idx = id2idx.get(item_id)
            if parts is None or idx is None:
                ranks.append(10**9)
                zero_overlap += 1
                continue
            base, loc, pop = parts
            final = base + loc * loc_boost + pop * pop_boost
            ranks.append(int((final > final[idx]).sum()) + 1)
            zero_overlap += int(base[idx] == 0)
            loc_ok += int(loc[idx] > 0)
    r = np.array(ranks)
    share = lambda m: f"{m.mean():6.1%}"
    print("\n=== ДИАГНОСТИКА: где стоит нужное объявление среди всех ===")
    print(f"Пар (запрос, нужное объявление): {n_pairs}")
    print(f"  в топ-50 (это и есть recall):   {share(r <= 50)}")
    print(f"  места 51-200:                   {share((r > 50) & (r <= 200))}   <- можно достать ранжированием")
    print(f"  места 201-1000:                 {share((r > 200) & (r <= 1000))}")
    print(f"  дальше 1000:                    {share(r > 1000)}   <- лексикой почти не достать")
    print(f"  НЕТ ни одного общего слова:     {zero_overlap / n_pairs:6.1%}   <- потолок лексики = {1 - zero_overlap / n_pairs:.1%}")
    print(f"  локация объявления = локации запроса: {loc_ok / n_pairs:6.1%}")


def dump_examples(generator, queries_df, relevant, loc_boost, pop_boost, quota=0,
                  n_miss=40, n_hit=15, path="examples_val.csv"):
    """Примеры: запрос, нужные объявления и что мы вернули вместо них."""
    titles = generator.items_df["item_title_raw"].values
    id2idx = {i: k for k, i in enumerate(generator.item_ids)}
    order = list(queries_df.index)
    np.random.default_rng(1).shuffle(order)
    rows, misses, hits = [], 0, 0
    for qkey in order:
        if misses >= n_miss and hits >= n_hit:
            break
        qrow = queries_df.loc[qkey]
        parts = generator.score_parts(qrow)
        if parts is None:
            continue
        base, loc, pop = parts
        final = base + loc * loc_boost + pop * pop_boost
        top = generator.pick(base, loc, pop, TOP_K, loc_boost, pop_boost, quota)
        found = set(generator.item_ids[top]) & relevant[qkey]
        is_hit = len(found) > 0
        if (is_hit and hits >= n_hit) or (not is_hit and misses >= n_miss):
            continue
        hits += is_hit
        misses += not is_hit
        rel_txt = []
        for item_id in list(relevant[qkey])[:3]:
            k = id2idx.get(item_id)
            if k is not None:
                rank = int((final > final[k]).sum()) + 1
                rel_txt.append(f"[место {rank}] {titles[k]}")
        rows.append({
            "исход": "НАЙДЕНО" if is_hit else "ПРОМАХ",
            "запрос": qrow["search_query"],
            "фильтры": qrow.get("search_infm_params_text", ""),
            "нужные объявления": " || ".join(rel_txt),
            "мы вернули (топ-5)": " || ".join(f"{titles[k]} ({final[k]:.2f})" for k in top[:5]),
        })
    out = pd.DataFrame(rows).sort_values("исход", ascending=False)
    out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"\n=== ПРИМЕРЫ (всего {len(out)}, сохранено в {path}) ===")
    for _, r in out[out["исход"] == "ПРОМАХ"].head(8).iterrows():
        print(f"\nЗАПРОС: {r['запрос']!r}   фильтры: {r['фильтры']!r}")
        print(f"  нужное:  {r['нужные объявления']}")
        print(f"  вернули: {r['мы вернули (топ-5)']}")

## `run_experiments` — подбор параметров

Три этапа:

1. **Сетка `pop_w × loc_w`** при `QUERY_USE_FILTERS ∈ {True, False}` — 56 конфигов, ~минуты.
2. **Сетка `TEXT_QUOTA`** при лучших `pop_w, loc_w`.
3. **Диагностика** + **выгрузка примеров** в `examples_val.csv`.

Возвращает лучшее `(use_filters, pop_w, loc_w, best_quota)` — их переносим в блок настроек.

In [36]:
def run_experiments(bench_items_df=None, n_queries=1000,
                    pop_grid=(0, 0.05, 0.1, 0.2),
                    loc_grid=(0, 0.1, 0.2, 0.35, 0.5, 1, 1000)):
    """Подбор параметров: сетка pop × loc, квота, диагностика, примеры."""
    items_pool, query_features, relevant = prepare_validation(bench_items_df)
    queries = query_features.iloc[:n_queries]
    print(f"\nЛемматизация: {'ВКЛ' if LEMMATIZE else 'ВЫКЛ'}; запросов: {len(queries)}")
    generator = BM25CandidateGenerator(items_pool)

    best = (-1.0, None)
    for use_filters in (True, False):
        t0 = time.time()
        table = evaluate_grid(generator, queries, relevant, pop_grid, loc_grid, use_filters)
        print(f"\n--- Recall@{TOP_K}; фильтры в запросе: {'ДА' if use_filters else 'НЕТ'} "
              f"({time.time() - t0:.0f} сек) ---")
        print(table.round(4).to_string())
        for p in pop_grid:
            for l in loc_grid:
                v = table.loc[f"pop={p}", f"loc={l}"]
                if v > best[0]:
                    best = (v, (use_filters, p, l))

    score, (use_filters, p, l) = best
    print(f"\n>>> ЛУЧШЕЕ: Recall@{TOP_K}={score:.4f} при QUERY_USE_FILTERS={use_filters}, "
          f"POPULARITY_BOOST={p}, LOCATION_BOOST={l}")

    quota_grid = (0, 5, 10, 15, 20, 25)
    qres = evaluate_quota(generator, queries, relevant, l, p, quota_grid, use_filters)
    print(f"\n--- Recall@{TOP_K} при разных TEXT_QUOTA (LOCATION_BOOST={l}, POPULARITY_BOOST={p}) ---")
    for q in quota_grid:
        print(f"  TEXT_QUOTA={q:>2}: {qres[q]:.4f}")
    best_q = max(qres, key=qres.get)
    print(f">>> ЛУЧШАЯ КВОТА: TEXT_QUOTA={best_q}  (Recall@{TOP_K}={qres[best_q]:.4f})")
    print("    Перенеси в блок настроек: QUERY_USE_FILTERS, LOCATION_BOOST, POPULARITY_BOOST, TEXT_QUOTA.")

    generator.use_filters = use_filters
    run_diagnostics(generator, queries, relevant, l, p)
    dump_examples(generator, queries, relevant, l, p, best_q)
    return best + (best_q,)

## Финальное предсказание

Строит `answer.csv` для `benchmark_queries` по параметрам из **блока настроек**.

In [37]:
def run_final_prediction():
    """Строит answer.csv для benchmark_queries / benchmark_items."""
    bench_items = pd.read_parquet(BENCH_ITEMS_PATH)
    bench_queries = pd.read_parquet(BENCH_QUERIES_PATH)

    generator = BM25CandidateGenerator(bench_items)

    query_ids, answers = [], []
    for _, qrow in bench_queries.iterrows():
        top_items = generator.search(qrow, top_k=TOP_K)
        query_ids.append(qrow["query_id"])
        answers.append(" ".join(top_items))

    answer_df = pd.DataFrame({"query_id": query_ids, "answer": answers})
    answer_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    print(f"Готово: {OUTPUT_PATH} ({len(answer_df)} строк)")

## Валидация `answer.csv`

Проверяет формат перед отправкой:
- колонки ровно `['query_id', 'answer']`, без индекса;
- все `query_id` из `benchmark_queries` присутствуют, лишних нет;
- нет дублей `query_id`;
- в каждой строке ≤ 50 `item_id`, без повторов;
- все `item_id` есть в `benchmark_items`.

`dtype=str` при чтении — обязательно: pandas может интерпретировать ID как числа и потерять ведущие нули/буквы.

**`inspect_final_answers`** выгружает превью того, что реально вернул поиск — удобно посмотреть глазами.

In [38]:
def validate_answer_csv(bench_queries: pd.DataFrame, bench_items: pd.DataFrame) -> None:
    """Проверка answer.csv по требованиям задания."""
    df = pd.read_csv(OUTPUT_PATH, dtype=str)
    problems = []

    if list(df.columns) != ["query_id", "answer"]:
        problems.append(f"Колонки должны быть ['query_id', 'answer'], а не {list(df.columns)}")

    expected_qids = set(bench_queries["query_id"].astype(str))
    got_qids = set(df["query_id"])
    if expected_qids - got_qids:
        problems.append(f"Пропущено {len(expected_qids - got_qids)} query_id")
    if got_qids - expected_qids:
        problems.append(f"Лишних query_id: {len(got_qids - expected_qids)}")
    if df["query_id"].duplicated().any():
        problems.append("Есть повторяющиеся query_id")

    valid_item_ids = set(bench_items["item_id"].astype(str))
    for qid, answer in zip(df["query_id"], df["answer"].fillna("")):
        items = answer.split() if answer else []
        if len(items) > 50:
            problems.append(f"{qid}: больше 50 item_id ({len(items)})")
        if len(items) != len(set(items)):
            problems.append(f"{qid}: повторы item_id внутри строки")
        unknown = [it for it in items if it not in valid_item_ids]
        if unknown:
            problems.append(f"{qid}: {len(unknown)} несуществующих item_id, напр. {unknown[0]!r}")
        if len(problems) > 30:
            break

    if problems:
        print(f"НАЙДЕНЫ ПРОБЛЕМЫ ({len(problems)}):")
        for p in problems[:30]:
            print(" -", p)
    else:
        print("Формат answer.csv в порядке.")


def inspect_final_answers(n=10, top=5, seed=0, path="answers_preview.csv"):
    """Показывает, что вернул поиск для benchmark-запросов."""
    ans = pd.read_csv(OUTPUT_PATH, dtype=str)
    bq = pd.read_parquet(BENCH_QUERIES_PATH,
                         columns=["query_id", "search_query", "search_infm_params_text"])
    bi = pd.read_parquet(BENCH_ITEMS_PATH, columns=["item_id", "item_title_raw"])
    title = dict(zip(bi["item_id"], bi["item_title_raw"]))
    df = bq.merge(ans, on="query_id")
    tops = [[title.get(i, "?") for i in str(a).split()[:top]] for a in df["answer"]]
    preview = pd.DataFrame({"query_id": df["query_id"], "запрос": df["search_query"],
                            "фильтры": df["search_infm_params_text"]})
    for k in range(top):
        preview[f"топ{k + 1}"] = [t[k] if k < len(t) else "" for t in tops]
    preview.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Полный просмотр всех {len(preview)} запросов сохранён в {path}\n")
    for _, r in preview.sample(min(n, len(preview)), random_state=seed).iterrows():
        print(f"ЗАПРОС: {r['запрос']!r}   фильтры: {r['фильтры']!r}")
        for k in range(top):
            print(f"   {k + 1}. {r[f'топ{k + 1}']}")
        print()

## Запуск

Поставь флаги под свой сценарий:

- **`DO_EXPERIMENTS = True`** — подбор параметров (~10 мин)
- **`DO_FINAL = True`** — построить `answer.csv`


In [ ]:
if __name__ == "__main__" or True:
    DO_EXPERIMENTS = False    
    DO_FINAL = True           

    # 0. Проверка входных данных (типы, формат item_id, пропуски)
    _queries_preview = pd.read_parquet(BENCH_QUERIES_PATH)
    _items_preview = pd.read_parquet(BENCH_ITEMS_PATH)
    sanity_check_inputs(_queries_preview, _items_preview)

    # (опционально) Сравнение с лемматизацией / без:
    # run_local_validation(bench_items_df=_items_preview)

    if DO_EXPERIMENTS:
        run_experiments(bench_items_df=_items_preview)

    if DO_FINAL:
        run_final_prediction()
        validate_answer_csv(_queries_preview, _items_preview)
        inspect_final_answers()

=== Проверка колонок ===
Колонки queries: ['query_id', 'search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
Колонки items:   ['item_title_raw', 'item_rating_reviews_count', 'item_rating', 'item_price', 'item_microcat_id', 'item_longitude', 'item_location_id', 'item_latitude', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_infm_params_text', 'item_id', 'item_description_raw', 'item_category_id']
search_category (int64)  vs  item_category_id (int64)  -> OK
search_location_id (int64)  vs  item_location_id (int64)  -> OK

=== Примеры сырого текста ===
search_query: ['перевозки владикавказ тбилиси', 'обзвон по базе', 'липоредукция подбородка', 'подъемник 4 х стоечный', 'монтаж видеодомофонов']
item_title_raw: ['Ремонт/выкуп компьют. и ноутбуков с выездом на дом', 'Обучение ребенка чтению', 'Афрокудри 5+', 'Монтаж малых архитектурных форм', 'Репетитор по истории 10 класс ЕГЭ']

=== Доля пропусков (NaN) в текстовых полях 